<a href="https://colab.research.google.com/github/KinzaAsif2456/discoverey/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I’m going with Logistic Regression for the first pass. Clustering doesn't make sense since we have labels, and I want to keep things interpretable before jumping into tree ensembles. I want to see if the baseline features actually hold up in a linear model before adding complexity. If LR can't beat the baseline, it's probably a feature engineering issue, not a model issue.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [106]:
#All imports
import os
import subprocess
import numpy as np
import pandas as pd
import duckdb
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance


In [76]:

%pip install -q duckdb huggingface_hub

In [77]:
# Step 1 — Fetch data from starter repo

STARTER_REPO = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
if not os.path.isdir("flyrank-ml-internship-starter"):
    subprocess.run(["git", "clone", "--depth", "1", STARTER_REPO, "flyrank-ml-internship-starter"], check=True)


df = pd.read_csv("flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")


In [78]:
# Step 2 —  Basic dataset checks
print(f"Dataset shape: {df.shape}")
print(f"Duplicate content_ids: {df['content_id'].duplicated().sum()}")
print(f"Overall decline rate: {df['trend_direction'].eq('down').mean():.3f}")
print("duplicate content_id rows:", df["content_id"].duplicated().sum())

Dataset shape: (30000, 44)
Duplicate content_ids: 0
Overall decline rate: 0.542
duplicate content_id rows: 0


In [79]:
# Step 3 — rebuild label and baseline signals
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

STALE_TIERS = ["91-180", "181+"]
CTR_THRESHOLD = 0.005
IMPRESSION_DECOY_LEVEL = 5000

df["stale_flag"] = df["freshness_tier"].isin(STALE_TIERS)
df["low_ctr_flag"] = (df["avg_position"] <= 20) & (df["ctr"] < CTR_THRESHOLD) & (df["impressions_90d"] >= 500)
df["is_decoy"] = (df["freshness_tier"] == "181+") & (df["impressions_90d"] >= IMPRESSION_DECOY_LEVEL)

def calculate_score(row):
    if row["is_decoy"]: return 3
    if row["stale_flag"] and row["low_ctr_flag"]: return 2
    if row["stale_flag"] or row["low_ctr_flag"]: return 1
    return 0

df["baseline_score"] = df.apply(calculate_score, axis=1)

First Attempt

In [80]:
# Step 4 — the split


train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["is_declining_label"]
)
print(train_df.shape, test_df.shape)



(24000, 49) (6000, 49)


In [81]:
# Step 5 — evaluate baseline on test_df
def precision_at_k(sub_df, score_col, k=50, tiebreak_col="impressions_90d"):
    top_k = sub_df.sort_values([score_col, tiebreak_col], ascending=[False, False]).head(k)
    return top_k["is_declining_label"].mean()

baseline_p50 = precision_at_k(test_df, "baseline_score", k=50)
print(f"Baseline Precision@50 (test set only): {baseline_p50:.3f}")

Baseline Precision@50 (test set only): 0.880


In [82]:
# Where did the biggest client end up in your current split?
top_client = df["client_id"].value_counts().idxmax()
print("biggest client:", top_client, "-> in test set:", top_client in test_df["client_id"].values)

# Check stability: with only 32 groups, one random split can be misleading.
# Run several seeds and see how much Precision@50 actually swings.
for seed in [0, 1, 42, 100, 7]:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_idx, te_idx = next(gss.split(df, groups=df["client_id"]))
    te = df.iloc[te_idx]
    p50 = precision_at_k(te, "baseline_score", k=50)
    print(f"seed={seed:>3}  test_clients={te['client_id'].nunique():>2}  test_rows={len(te):>5}  precision@50={p50:.3f}")

biggest client: client_19581e27de -> in test set: True
seed=  0  test_clients= 7  test_rows=10179  precision@50=0.780
seed=  1  test_clients= 7  test_rows= 2162  precision@50=0.860
seed= 42  test_clients= 7  test_rows= 6163  precision@50=0.700
seed=100  test_clients= 7  test_rows= 3713  precision@50=0.600
seed=  7  test_clients= 7  test_rows=11754  precision@50=0.780


In [83]:
# How many content items does each client actually own?
print("unique clients:", df["client_id"].nunique())
print(df["client_id"].value_counts().describe())

# Also worth knowing before interpreting that 0.880 number later:
print("overall decline rate:", df["is_declining_label"].mean())

unique clients: 32
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
Name: count, dtype: float64
overall decline rate: 0.5420666666666667


In [84]:

n_splits = 5
gkf = GroupKFold(n_splits=n_splits)

baseline_p50s = []
for fold, (train_idx, test_idx) in enumerate(gkf.split(df, groups=df["client_id"])):
    test_fold = df.iloc[test_idx]
    p50 = precision_at_k(test_fold, "baseline_score", k=50)
    baseline_p50s.append(p50)
    print(f"fold {fold}: test_clients={test_fold['client_id'].nunique():>2}  "
          f"test_rows={len(test_fold):>5}  precision@50={p50:.3f}")

print(f"\nBaseline Precision@50, {n_splits}-fold client CV: "
      f"{np.mean(baseline_p50s):.3f} ± {np.std(baseline_p50s):.3f}")

fold 0: test_clients= 1  test_rows= 7008  precision@50=0.740
fold 1: test_clients= 7  test_rows= 5731  precision@50=0.960
fold 2: test_clients= 8  test_rows= 5753  precision@50=0.620
fold 3: test_clients= 8  test_rows= 5755  precision@50=0.900
fold 4: test_clients= 8  test_rows= 5753  precision@50=0.720

Baseline Precision@50, 5-fold client CV: 0.788 ± 0.124


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**STEP 1** —  Check for possible data leakage

In [85]:
# # Check for possible data leakage in delta columns before feature selection
df["_impr_delta"] = df["impressions_last_30d"] - df["impressions_prev_30d"]
print(df.groupby("is_declining_label")["_impr_delta"].describe())

# Check whether this feature alone predicts the label
from sklearn.metrics import roc_auc_score
print("AUC using impr delta alone:", roc_auc_score(df["is_declining_label"], -df["_impr_delta"]))

                      count        mean          std      min    25%    50%  \
is_declining_label                                                            
0                   13738.0  253.042728  3331.614776 -19024.0   -1.0    3.0   
1                   16262.0 -866.861026  2824.223534 -97995.0 -617.0 -158.0   

                     75%       max  
is_declining_label                  
0                   67.0  211258.0  
1                  -32.0      -1.0  
AUC using impr delta alone: 0.899071919995329


**STEP 2** — Feature set

Model A: same information the baseline rule used, nothing more. It is the real test — same inputs, does ML beat hand-tuned thresholds?

In [86]:
feature_A = ["freshness_tier_enc", "avg_position", "ctr", "impressions_90d"]

Model B (optional, only after Model A is fully working): baseline's inputs plus a couple of legitimate signals the baseline never used.

In [97]:
feature_B = feature_A + ["word_count", "engagement_rate"]

In [87]:

#  Make sure no leakage columns were added
# from the leakage check in Step A (trend_*, *_last_30d, *_prev_30d)
leak_terms = ["trend", "last_30d", "prev_30d"]
for cols, name in [(feature_A, "A"), (feature_B, "B")]:
    leak_columns = [c for c in cols if any(term in c for term in leak_terms)]
    print(f"Model {name} leak check:", "CLEAN" if not leak_columns else f"leak_columns: {leak_columns}")

Model A leak check: CLEAN
Model B leak check: CLEAN


**STEP 3**— Encode freshness_tier

In [88]:
tier_order = ["0-30", "31-90", "91-180", "181+"]
df["freshness_tier_enc"] = df["freshness_tier"].map({t: i for i, t in enumerate(tier_order)})
print(df[["freshness_tier", "freshness_tier_enc"]].drop_duplicates())

    freshness_tier  freshness_tier_enc
0             0-30                   0
9           91-180                   2
91            181+                   3
457          31-90                   1


**STEP 4** — Train and evaluate the model using GroupKFold

In [108]:
#1. Feature Engineering: Create missing flag for zero/unranked position
df["avg_position_missing"] = (df["avg_position"] == 0).astype(int)

feature_cols = [
    "freshness_tier_enc",
    "avg_position",
    "avg_position_missing",
    "ctr",
    "impressions_90d"
]

# Track metrics across folds
model_p50s = []
auc_scores = []

In [109]:
# 2. Cross-Validation Loop
for fold, (train_idx, test_idx) in enumerate(gkf.split(df, groups=df["client_id"])):
    train_fold, test_fold = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

    X_train, y_train = train_fold[feature_cols], train_fold["is_declining_label"]
    X_test, y_test = test_fold[feature_cols], test_fold["is_declining_label"]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", random_state=42)),
    ])
    model.fit(X_train, y_train)

    # Predict probabilities for ROC-AUC & Precision@50
    test_fold["model_score"] = model.predict_proba(X_test)[:, 1]

    # Evaluate Precision@50 & ROC-AUC for this fold
    p50 = precision_at_k(test_fold, "model_score", k=50)
    auc = roc_auc_score(y_test, test_fold["model_score"])

    model_p50s.append(p50)
    auc_scores.append(auc)

    print(
        f"fold {fold}: test_clients={test_fold['client_id'].nunique():>2}  "
        f"test_rows={len(test_fold):>5}  precision@50={p50:.3f}  roc_auc={auc:.3f}"
    )

fold 0: test_clients= 1  test_rows= 7008  precision@50=0.920  roc_auc=0.641
fold 1: test_clients= 7  test_rows= 5731  precision@50=0.740  roc_auc=0.543
fold 2: test_clients= 8  test_rows= 5753  precision@50=0.580  roc_auc=0.671
fold 3: test_clients= 8  test_rows= 5755  precision@50=0.760  roc_auc=0.542
fold 4: test_clients= 8  test_rows= 5753  precision@50=0.400  roc_auc=0.503


In [110]:
# 3. Aggregate Performance Summary
print(
    f"\nModel Precision@50, {n_splits}-fold client CV: "
    f"{np.mean(model_p50s):.3f} ± {np.std(model_p50s):.3f}"
)
print(
    f"Model ROC-AUC, {n_splits}-fold client CV: "
    f"{np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}"
)


Model Precision@50, 5-fold client CV: 0.680 ± 0.177
Model ROC-AUC, 5-fold client CV: 0.580 ± 0.064


In [111]:
# 4. Permutation Importance (evaluated on the final fold)
perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42
)

perm_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": perm.importances_mean
}).sort_values("Importance", ascending=False)

print("\nPermutation Importance:")
print(perm_df.to_string(index=False))


Permutation Importance:
             Feature  Importance
avg_position_missing    0.024996
        avg_position    0.022840
                 ctr    0.002364
     impressions_90d   -0.014775
  freshness_tier_enc   -0.018477


In [112]:
# 5. Final Results Comparison Table
results = pd.DataFrame({
    "Method": [
        "Baseline (rule-based)",
        "Logistic Regression"
    ],
    "Precision@50 (Mean ± Std)": [
        f"{np.mean(baseline_p50s):.3f} ± {np.std(baseline_p50s):.3f}",
        f"{np.mean(model_p50s):.3f} ± {np.std(model_p50s):.3f}"
    ],
    "ROC-AUC": [
        "N/A",  # Baseline outputs deterministic ranking rules, not soft probabilities
        f"{np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}"
    ]
})

results

,Method,Precision@50 (Mean ± Std),ROC-AUC
0,Baseline (rule-based),0.788 ± 0.124,N/A
1,Logistic Regression,0.680 ± 0.177,0.580 ± 0.064


In [101]:
df["avg_position_missing"] = (df["avg_position"] == 0).astype(int)

In [102]:
feature_cols = [
    "freshness_tier_enc",
    "avg_position",
    "avg_position_missing",
    "ctr",
    "impressions_90d"
]

In [103]:

model_p50s = []
for fold, (train_idx, test_idx) in enumerate(gkf.split(df, groups=df["client_id"])):
    train_fold, test_fold = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

    X_train, y_train = train_fold[feature_cols], train_fold["is_declining_label"]
    X_test = test_fold[feature_cols]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", random_state=42)),
    ])
    model.fit(X_train, y_train)
    test_fold["model_score"] = model.predict_proba(X_test)[:, 1]

    p50 = precision_at_k(test_fold, "model_score", k=50)
    model_p50s.append(p50)
    print(f"fold {fold}: test_clients={test_fold['client_id'].nunique():>2}  "
          f"test_rows={len(test_fold):>5}  precision@50={p50:.3f}")

print(f"\nModel Precision@50, {n_splits}-fold client CV: "
      f"{np.mean(model_p50s):.3f} ± {np.std(model_p50s):.3f}")

fold 0: test_clients= 1  test_rows= 7008  precision@50=0.920
fold 1: test_clients= 7  test_rows= 5731  precision@50=0.740
fold 2: test_clients= 8  test_rows= 5753  precision@50=0.580
fold 3: test_clients= 8  test_rows= 5755  precision@50=0.760
fold 4: test_clients= 8  test_rows= 5753  precision@50=0.400

Model Precision@50, 5-fold client CV: 0.680 ± 0.177


**STEP 5** — the comparison table

In [104]:
print(f"\nModel Precision@50, {n_splits}-fold client CV: "
      f"{np.mean(model_p50s):.3f} ± {np.std(model_p50s):.3f}")


Model Precision@50, 5-fold client CV: 0.680 ± 0.177


In [105]:
results = pd.DataFrame({
    "method": ["Baseline (rule-based)", "Logistic Regression"],
    "precision_at_50_mean": [np.mean(baseline_p50s), np.mean(model_p50s)],
    "precision_at_50_std":  [np.std(baseline_p50s),  np.std(model_p50s)],
})
results

,method,precision_at_50_mean,precision_at_50_std
0,Baseline (rule-based),0.788,0.124322
1,Logistic Regression,0.680,0.176635


**STEP 5-** Observation

With zero client leakage across folds, the baseline beat the model by a wide margin: 0.788 ± 0.124 versus 0.496 ± 0.182.

The baseline performed better because its rules already include conditions like minimum impressions and special handling for missing values. Logistic Regression treated all features as normal numeric values, so it could not capture those rules as well.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [91]:
model_p50s = []
coef_list = []
fold_results = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(df, groups=df["client_id"])):
    train_fold, test_fold = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

    X_train, y_train = train_fold[feature_cols], train_fold["is_declining_label"]
    X_test = test_fold[feature_cols]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", random_state=42)),
    ])
    model.fit(X_train, y_train)
    test_fold["model_score"] = model.predict_proba(X_test)[:, 1]

    # Top 50 predictions from the model
    ranked = test_fold.sort_values(["model_score", "impressions_90d"], ascending=[False, False])
    test_fold["in_top50"] = test_fold["content_id"].isin(ranked.head(50)["content_id"])

    # Baseline top 50 on the same fold
    baseline_ranked= test_fold.sort_values(["baseline_score", "impressions_90d"], ascending=[False, False])
    test_fold["baseline_in_top50"] = test_fold["content_id"].isin(baseline_ranked.head(50)["content_id"])

    test_fold["fold"] = fold
    p50 = test_fold.loc[test_fold["in_top50"], "is_declining_label"].mean()
    model_p50s.append(p50)
    coef_list.append(dict(zip(feature_cols, model.named_steps["clf"].coef_[0])))
    fold_results.append(test_fold)

oof = pd.concat(fold_results, ignore_index=True)
coef_df = pd.DataFrame(coef_list)
print(f"Model Precision@50: {np.mean(model_p50s):.3f} ± {np.std(model_p50s):.3f}")

Model Precision@50: 0.496 ± 0.182


In [92]:
print("Mean coefficient per feature (on standardized scale, across 5 folds):")
print(coef_df.mean().sort_values(key=abs, ascending=False))
print("\nfold-to-fold coefficient stability:")
print(coef_df.std())

Mean coefficient per feature (on standardized scale, across 5 folds):
ctr                  -0.195149
freshness_tier_enc    0.182024
avg_position         -0.087778
impressions_90d      -0.064696
dtype: float64

fold-to-fold coefficient stability:
freshness_tier_enc    0.056157
avg_position          0.067993
ctr                   0.020082
impressions_90d       0.038081
dtype: float64


In [93]:
false_positives = oof[oof["in_top50"] & (oof["is_declining_label"] == 0)]
false_negatives = oof[~oof["in_top50"] & (oof["is_declining_label"] == 1)]
print(f"False positives: {len(false_positives)} | Missed decliners: {len(false_negatives)} / {oof['is_declining_label'].sum()}")

print("\nAgreement between model and baseline on who makes top-50:")
print(pd.crosstab(oof["in_top50"], oof["baseline_in_top50"], rownames=["model_top50"], colnames=["baseline_top50"]))

display_cols = ["content_id", "client_id", "freshness_tier", "avg_position", "ctr",
               "impressions_90d", "model_score", "baseline_score", "is_declining_label"]
print("\nSample false positives (model over-trusted these):")
print(false_positives[display_cols].sort_values("model_score", ascending=False).head(8).to_string(index=False))

False positives: 126 | Missed decliners: 16138 / 16262

Agreement between model and baseline on who makes top-50:
baseline_top50  False  True 
model_top50                 
False           29512    238
True              238     12

Sample false positives (model over-trusted these):
          content_id         client_id freshness_tier  avg_position  ctr  impressions_90d  model_score  baseline_score  is_declining_label
content_06e19c6486b0 client_4ec9599fc2           181+           5.0 0.00               10     0.683549               1                   0
content_ab27c30d81f4 client_4ec9599fc2           181+           8.9 0.00              103     0.679522               1                   0
content_607ea6e3f876 client_4ec9599fc2           181+          20.4 0.00               64     0.667942               1                   0
content_bdbec75c1148 client_7f2253d7e2           181+          21.8 0.15             1316     0.662665               1                   0
content_6476d1d8c050 cl

In [94]:
# 1) Is avg_position == 0 a real "position 0" or a sentinel for "no ranking data"?
# A linear model has no way to know 0 is special — it'll treat it as "best possible position."
print(df["avg_position"].value_counts().head(5))
print("decline rate when avg_position==0:", df.loc[df["avg_position"] == 0, "is_declining_label"].mean())
print("overall decline rate:", df["is_declining_label"].mean())

# 2) impressions_90d — how skewed/heavy-tailed is it? StandardScaler assumes
# roughly symmetric data; a power-law column can badly distort a linear boundary.
print(df["impressions_90d"].describe())

avg_position
0.0    1205
7.0     263
6.0     256
5.0     249
8.0     245
Name: count, dtype: int64
decline rate when avg_position==0: 0.006639004149377593
overall decline rate: 0.5420666666666667
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64


In [98]:
# Create a binary feature for missing ranking positions
df["avg_position_missing"] = (df["avg_position"] == 0).astype(int)

print("\nDecline rate by missing position flag:")
print(df.groupby("avg_position_missing")["is_declining_label"].mean())


Decline rate by missing position flag:
avg_position_missing
0    0.564473
1    0.006639
Name: is_declining_label, dtype: float64


In [95]:
# Confirm strict client isolation across folds
for fold, (train_idx, test_idx) in enumerate(gkf.split(df, groups=df["client_id"])):
    train_clients = set(df.iloc[train_idx]["client_id"])
    test_clients = set(df.iloc[test_idx]["client_id"])
    assert not (train_clients & test_clients), f"Leakage detected in fold {fold}!"

print("Assertion passed: Zero client overlap across cross-validation folds.")

Assertion passed: Zero client overlap across cross-validation folds.


**Interpretation based upon the analysis**

The model gave the highest weight to CTR (coefficient -0.195) and freshness tier (+0.182). This matches what I expected because older pages with lower CTR are usually more likely to decline. The coefficients stayed fairly consistent across the five folds (σ ≤ 0.068), which suggests the model learned similar feature relationships regardless of which clients were included in training.

One problem is that the model only ranks pages using predicted probability. Because of this, some low-traffic pages received high scores even though they were not important declining pages. The model scored them without considering a minimum impression threshold. For instance, it flagged content_06e19c6486b0 as a top false positive (scoring it 0.683) simply because it was old (181+ days) with a 0% CTR—completely ignoring the fact that it only had 10 total impressions. The baseline avoids this trap by explicitly requiring at least 500 impressions before triggering its  low-CTR flag; the linear model simply doesn't have an equivalent rule.

I tested adding an avg_position_missing feature to separate missing ranking values from genuine ranking positions. This improved the model's Precision@50 from 0.496 to 0.680, showing that treating avg_position == 0 as a normal numeric value was reducing the model's performance. Although the Logistic Regression model still did not outperform the rule-based baseline, this feature provided a more accurate representation of the data.


In [96]:
print(results)

                  method  precision_at_50_mean  precision_at_50_std
0  Baseline (rule-based)                 0.788             0.124322
1    Logistic Regression                 0.496             0.182165


**Evidence**

**Result:**
The baseline still wins (0.788 vs 0.496).

**Why?** Logistic Regression is treating the features as linear, but the data is wonky. The impressions_90d values are highly skewed, which makes it difficult for Logistic Regression to model the relationship properly, and the model is being too aggressive on old content that doesn't have enough traffic to actually be 'declining.'

**Next steps:** I need to handle the avg_position == 0 case—maybe with a binary 'is_missing' flag. I also should probably log-transform the impressions or move to a Decision Tree to handle the non-linear relationship between volume and decline risk. Adding the avg_position_missing feature noticeably improved the Logistic Regression model, confirming that avg_position == 0 should not be treated as a genuine ranking position. However, the rule-based baseline still achieved a higher Precision@50 (0.788 vs. 0.680), suggesting that its manually designed rules continue to capture important patterns that the linear model misses.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.